# Code for Streamlit Dashboard

## 1. Data Generation

Generate a synthetic retail transactions dataset: 6,000 orders across 2023–2024, with seasonal shopping patterns (holiday surges, weekend spikes, back-to-school bumps), realistic price distributions per category, customer segments, sales channels, payment methods, shipping costs, and order status (Completed / Returned / Cancelled).
<br><br>
Saves the dataset as `retail_sales.csv`, which the dashboard reads and visualizes.

In [11]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# -------------------------------------------------------------------
# Generate a more realistic retail sales dataset
# -------------------------------------------------------------------
np.random.seed(42)

N_ROWS = 6000
START_DATE = pd.Timestamp('2023-01-01')
END_DATE = pd.Timestamp('2024-12-31')
all_dates = pd.date_range(START_DATE, END_DATE, freq='D')

regions = ['North', 'South', 'East', 'West', 'Central']
region_states = {
    'North': ['Minnesota', 'Wisconsin', 'North Dakota'],
    'South': ['Texas', 'Georgia', 'Florida'],
    'East':  ['New York', 'New Jersey', 'Massachusetts'],
    'West':  ['California', 'Washington', 'Oregon'],
    'Central': ['Illinois', 'Ohio', 'Missouri'],
}

# category -> (subcategories, price range, base weight)
catalog = {
    'Electronics': {
        'subs': ['Headphones', 'Smartphones', 'Laptops', 'Cameras', 'Smart Home'],
        'price_range': (25, 1200),
        'weight': 0.22,
    },
    'Apparel': {
        'subs': ["Men's Wear", "Women's Wear", "Kids' Wear", 'Footwear', 'Accessories'],
        'price_range': (10, 150),
        'weight': 0.28,
    },
    'Home Goods': {
        'subs': ['Furniture', 'Kitchenware', 'Bedding', 'Decor', 'Storage'],
        'price_range': (15, 800),
        'weight': 0.20,
    },
    'Books': {
        'subs': ['Fiction', 'Non-Fiction', "Children's", 'Educational', 'Comics'],
        'price_range': (6, 45),
        'weight': 0.12,
    },
    'Sporting Goods': {
        'subs': ['Fitness Equipment', 'Outdoor Gear', 'Team Sports', 'Cycling', 'Footwear'],
        'price_range': (12, 500),
        'weight': 0.10,
    },
    'Beauty & Health': {
        'subs': ['Skincare', 'Haircare', 'Wellness', 'Makeup', 'Personal Care'],
        'price_range': (5, 120),
        'weight': 0.08,
    },
}

categories = list(catalog.keys())
category_weights = np.array([catalog[c]['weight'] for c in categories])
category_weights = category_weights / category_weights.sum()

segments = ['Consumer', 'Corporate', 'Home Office']
segment_weights = [0.62, 0.25, 0.13]

channels = ['Online', 'In-Store']
channel_weights = [0.58, 0.42]

payment_methods = ['Credit Card', 'Debit Card', 'PayPal', 'Gift Card', 'Cash']
payment_weights = [0.42, 0.24, 0.18, 0.08, 0.08]

order_statuses = ['Completed', 'Returned', 'Cancelled']
status_weights = [0.88, 0.09, 0.03]

def seasonal_weight(date):
    """Higher likelihood of purchase around holidays / weekends."""
    weight = 1.0
    month, day, weekday = date.month, date.day, date.dayofweek
    # Holiday shopping surge: Nov-Dec
    if month == 11 and day >= 20:
        weight *= 2.4
    elif month == 12 and day <= 24:
        weight *= 2.8
    elif month == 12:
        weight *= 1.3
    # Back-to-school bump
    if month in (8, 9):
        weight *= 1.25
    # Summer sale bump
    if month in (6, 7):
        weight *= 1.15
    # Weekends see more shopping traffic
    if weekday >= 5:
        weight *= 1.3
    return weight

date_weights = np.array([seasonal_weight(d) for d in all_dates])
date_weights = date_weights / date_weights.sum()
order_dates = np.random.choice(all_dates, size=N_ROWS, p=date_weights)

rows = []
for i in range(N_ROWS):
    region = np.random.choice(regions)
    state = np.random.choice(region_states[region])
    category = np.random.choice(categories, p=category_weights)
    info = catalog[category]
    subcategory = np.random.choice(info['subs'])
    low, high = info['price_range']
    # log-normal-ish spread keeps most prices modest with occasional high-ticket items
    unit_price = round(float(np.clip(np.random.lognormal(mean=np.log((low + high) / 3), sigma=0.5), low, high)), 2)

    units_sold = int(np.clip(np.random.poisson(2.4) + 1, 1, 15))
    discount = round(float(np.random.choice([0, 0, 0, 0.05, 0.1, 0.15, 0.2, 0.25],
                                             p=[0.35, 0.1, 0.1, 0.15, 0.1, 0.1, 0.05, 0.05])), 2)

    segment = np.random.choice(segments, p=segment_weights)
    channel = np.random.choice(channels, p=channel_weights)
    payment = np.random.choice(payment_methods, p=payment_weights)
    status = np.random.choice(order_statuses, p=status_weights)

    shipping_cost = 0.0 if channel == 'In-Store' else round(float(np.random.choice(
        [0, 4.99, 7.99, 12.99], p=[0.4, 0.3, 0.2, 0.1])), 2)

    rows.append({
        'order_id': f"ORD-{100000 + i}",
        'date': order_dates[i],
        'region': region,
        'state': state,
        'store_id': f"{region[:2].upper()}-{np.random.randint(1, 6):02d}",
        'customer_id': f"CUST-{np.random.randint(1, 1800):05d}",
        'customer_segment': segment,
        'product_category': category,
        'product_subcategory': subcategory,
        'units_sold': units_sold,
        'unit_price': unit_price,
        'discount': discount,
        'shipping_cost': shipping_cost,
        'payment_method': payment,
        'sales_channel': channel,
        'order_status': status,
    })

df_dummy = pd.DataFrame(rows).sort_values('date').reset_index(drop=True)
df_dummy.to_csv('retail_sales.csv', index=False)
print(f"Generated 'retail_sales.csv' with {len(df_dummy):,} realistic rows across "
      f"{df_dummy['date'].min().date()} to {df_dummy['date'].max().date()}.")


Generated 'retail_sales.csv' with 6,000 realistic rows across 2023-01-01 to 2024-12-31.


To run a Streamlit application in Google Colab, you typically need to use a tunneling service like `ngrok` to expose the local server to the internet.

Here are the steps:
1.  **Install `pyngrok`**: This Python wrapper helps manage `ngrok`.
2.  **Get `ngrok` Authtoken**: You'll need to sign up for a free `ngrok` account and get an authtoken from your dashboard. Add this authtoken to your Colab secrets (under the 🔑 icon on the left panel, named `NGROK_AUTH_TOKEN`).
3.  **Save your Streamlit code to a `.py` file**: The `streamlit run` command expects a Python file as an argument.
4.  **Run Streamlit with `ngrok`**: Use `pyngrok` to establish a tunnel and launch your Streamlit app.

<br>

Colab Secret Authtoken

Name: NGROK_AUTH_TOKEN

Value: already stored

## 2. Dashboard Deployment

Write the dashboard's visual interface and logic into a Python script `streamlit_app.py` and then run it in the background as a local web server.
<br><br>
The dashboard contains:
- **5 tabs** (Sales Trends, Product Analysis, Regional Analysis, Customer Insights, Raw Data)
- **6 KPI cards** (Total Revenue, Orders, Avg Order Value, Units Sold, Return Rate, Unique Customers)
- A **day-of-week** and **order-status** breakdown
- **Subcategory** and **store-level** treemap views
- **Customer segment** and **payment method** charts
- A **top-10 customers** table
- An **in-app search** over the filtered data
- A **CSV download** button — plus extra filters for customer segment, sales channel, order status, and an option to exclude returned/cancelled orders from revenue figures.
<br>

Then configure and launch a secure `ngrok` tunnel to map that local web server onto a public web URL. By clicking the generated public link, we can open and interact with the live retail sales dashboard right from our browser.

In [12]:
# Install pyngrok
!pip install -q pyngrok

# install streamlit
!pip install -q streamlit plotly

import os
from pyngrok import ngrok, conf
from google.colab import userdata

streamlit_app_code = """
import pandas as pd
import numpy as np
import streamlit as st
import plotly.express as px

# -------------------------------------------------------------------
# Page setup
# -------------------------------------------------------------------
st.set_page_config(
    page_title="Retail Sales Dashboard",
    page_icon="\\U0001F4CA",
    layout="wide",
)

st.title("Retail Sales Dashboard")
st.caption("Explore revenue, sales volume, customer behavior, and product performance.")

# -------------------------------------------------------------------
# Load & prepare data
# -------------------------------------------------------------------
@st.cache_data
def load_data():
    df = pd.read_csv("retail_sales.csv")
    df["date"] = pd.to_datetime(df["date"])
    df["revenue"] = df["units_sold"] * df["unit_price"] * (1 - df["discount"])
    df["net_revenue"] = df["revenue"] - df["shipping_cost"]
    df["order_value"] = df["revenue"]
    df["weekday"] = df["date"].dt.day_name()
    df["month"] = df["date"].dt.to_period("M").astype(str)
    return df

df = load_data()

# -------------------------------------------------------------------
# Sidebar filters
# -------------------------------------------------------------------
st.sidebar.header("Filters")

min_date, max_date = df["date"].min(), df["date"].max()
date_range = st.sidebar.date_input(
    "Date range", value=(min_date, max_date), min_value=min_date, max_value=max_date
)
if isinstance(date_range, tuple) and len(date_range) == 2:
    start_date, end_date = date_range
else:
    start_date, end_date = min_date, max_date

selected_regions = st.sidebar.multiselect(
    "Region", options=sorted(df["region"].unique()), default=sorted(df["region"].unique())
)
selected_categories = st.sidebar.multiselect(
    "Product category", options=sorted(df["product_category"].unique()),
    default=sorted(df["product_category"].unique())
)
selected_segments = st.sidebar.multiselect(
    "Customer segment", options=sorted(df["customer_segment"].unique()),
    default=sorted(df["customer_segment"].unique())
)
selected_channels = st.sidebar.multiselect(
    "Sales channel", options=sorted(df["sales_channel"].unique()),
    default=sorted(df["sales_channel"].unique())
)
selected_statuses = st.sidebar.multiselect(
    "Order status", options=sorted(df["order_status"].unique()),
    default=sorted(df["order_status"].unique())
)

st.sidebar.divider()
exclude_returns = st.sidebar.checkbox("Exclude returned/cancelled orders from revenue", value=True)

filtered_df = df[
    (df["region"].isin(selected_regions))
    & (df["product_category"].isin(selected_categories))
    & (df["customer_segment"].isin(selected_segments))
    & (df["sales_channel"].isin(selected_channels))
    & (df["order_status"].isin(selected_statuses))
    & (df["date"].between(pd.to_datetime(start_date), pd.to_datetime(end_date)))
].copy()

if exclude_returns:
    revenue_df = filtered_df[filtered_df["order_status"] == "Completed"]
else:
    revenue_df = filtered_df

if filtered_df.empty:
    st.warning("No data matches the selected filters. Try widening your selection.")
    st.stop()

# -------------------------------------------------------------------
# KPI row
# -------------------------------------------------------------------
total_revenue = revenue_df["revenue"].sum()
total_orders = filtered_df["order_id"].nunique()
avg_order_value = revenue_df["order_value"].mean() if len(revenue_df) else 0
total_units = revenue_df["units_sold"].sum()
return_rate = (filtered_df["order_status"] == "Returned").mean() * 100
unique_customers = filtered_df["customer_id"].nunique()

k1, k2, k3, k4, k5, k6 = st.columns(6)
k1.metric("Total Revenue", f"${total_revenue:,.0f}")
k2.metric("Orders", f"{total_orders:,}")
k3.metric("Avg Order Value", f"${avg_order_value:,.2f}")
k4.metric("Units Sold", f"{total_units:,}")
k5.metric("Return Rate", f"{return_rate:.1f}%")
k6.metric("Unique Customers", f"{unique_customers:,}")

st.divider()

# -------------------------------------------------------------------
# Tabs for the different analysis views
# -------------------------------------------------------------------
tab_trends, tab_products, tab_regions, tab_customers, tab_data = st.tabs(
    ["Sales Trends", "Product Analysis", "Regional Analysis", "Customer Insights", "Raw Data"]
)

# --- Sales Trends -----------------------------------------------------
with tab_trends:
    col1, col2 = st.columns((2, 1))

    with col1:
        monthly_revenue = (
            revenue_df.set_index("date").resample("ME")["revenue"].sum().reset_index()
        )
        fig = px.line(monthly_revenue, x="date", y="revenue", markers=True,
                       title="Monthly Revenue Trend")
        st.plotly_chart(fig, use_container_width=True)

    with col2:
        channel_rev = revenue_df.groupby("sales_channel")["revenue"].sum().reset_index()
        fig = px.pie(channel_rev, names="sales_channel", values="revenue",
                      title="Revenue by Sales Channel", hole=0.45)
        st.plotly_chart(fig, use_container_width=True)

    col3, col4 = st.columns(2)
    with col3:
        weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
        weekday_rev = (
            revenue_df.groupby("weekday")["revenue"].sum()
            .reindex(weekday_order).reset_index()
        )
        fig = px.bar(weekday_rev, x="weekday", y="revenue", title="Revenue by Day of Week")
        st.plotly_chart(fig, use_container_width=True)

    with col4:
        status_counts = filtered_df["order_status"].value_counts().reset_index()
        status_counts.columns = ["order_status", "count"]
        fig = px.bar(status_counts, x="order_status", y="count", color="order_status",
                      title="Order Status Breakdown")
        st.plotly_chart(fig, use_container_width=True)

# --- Product Analysis ---------------------------------------------------
with tab_products:
    col1, col2 = st.columns(2)

    with col1:
        cat_rev = revenue_df.groupby("product_category")["revenue"].sum().sort_values(ascending=False).reset_index()
        fig = px.bar(cat_rev, x="product_category", y="revenue", title="Revenue by Category")
        st.plotly_chart(fig, use_container_width=True)

    with col2:
        sub_rev = (
            revenue_df.groupby(["product_category", "product_subcategory"])["revenue"]
            .sum().reset_index().sort_values("revenue", ascending=False).head(10)
        )
        fig = px.bar(sub_rev, x="revenue", y="product_subcategory", color="product_category",
                      orientation="h", title="Top 10 Subcategories by Revenue")
        fig.update_layout(yaxis=dict(categoryorder="total ascending"))
        st.plotly_chart(fig, use_container_width=True)

    discount_impact = (
        revenue_df.groupby("product_category")
        .agg(avg_discount=("discount", "mean"), revenue=("revenue", "sum"))
        .reset_index()
    )
    fig = px.scatter(discount_impact, x="avg_discount", y="revenue", size="revenue",
                      color="product_category", title="Average Discount vs Revenue by Category")
    st.plotly_chart(fig, use_container_width=True)

# --- Regional Analysis ---------------------------------------------------
with tab_regions:
    col1, col2 = st.columns(2)

    with col1:
        region_rev = revenue_df.groupby("region")["revenue"].sum().sort_values(ascending=False).reset_index()
        fig = px.bar(region_rev, x="region", y="revenue", title="Revenue by Region")
        st.plotly_chart(fig, use_container_width=True)

    with col2:
        state_rev = revenue_df.groupby("state")["revenue"].sum().sort_values(ascending=False).head(10).reset_index()
        fig = px.bar(state_rev, x="revenue", y="state", orientation="h",
                      title="Top 10 States by Revenue")
        fig.update_layout(yaxis=dict(categoryorder="total ascending"))
        st.plotly_chart(fig, use_container_width=True)

    store_rev = revenue_df.groupby(["region", "store_id"])["revenue"].sum().reset_index()
    fig = px.treemap(store_rev, path=["region", "store_id"], values="revenue",
                      title="Revenue by Region and Store")
    st.plotly_chart(fig, use_container_width=True)

# --- Customer Insights ---------------------------------------------------
with tab_customers:
    col1, col2 = st.columns(2)

    with col1:
        segment_rev = revenue_df.groupby("customer_segment")["revenue"].sum().reset_index()
        fig = px.pie(segment_rev, names="customer_segment", values="revenue",
                      title="Revenue by Customer Segment", hole=0.45)
        st.plotly_chart(fig, use_container_width=True)

    with col2:
        payment_rev = revenue_df.groupby("payment_method")["revenue"].sum().sort_values(ascending=False).reset_index()
        fig = px.bar(payment_rev, x="payment_method", y="revenue", title="Revenue by Payment Method")
        st.plotly_chart(fig, use_container_width=True)

    top_customers = (
        revenue_df.groupby("customer_id")
        .agg(total_spent=("revenue", "sum"), orders=("order_id", "nunique"))
        .sort_values("total_spent", ascending=False).head(10).reset_index()
    )
    st.subheader("Top 10 Customers by Spend")
    st.dataframe(top_customers, use_container_width=True, hide_index=True)

# --- Raw Data ---------------------------------------------------
with tab_data:
    st.subheader("Filtered Transaction Data")
    search = st.text_input("Search (matches any column as text)")
    display_df = filtered_df.copy()
    if search:
        mask = display_df.apply(lambda col: col.astype(str).str.contains(search, case=False, na=False))
        display_df = display_df[mask.any(axis=1)]

    st.dataframe(display_df.sort_values("date", ascending=False), use_container_width=True, hide_index=True)
    st.caption(f"Showing {len(display_df):,} of {len(filtered_df):,} filtered rows.")

    csv_bytes = display_df.to_csv(index=False).encode("utf-8")
    st.download_button(
        "Download filtered data as CSV",
        data=csv_bytes,
        file_name="filtered_retail_sales.csv",
        mime="text/csv",
    )
"""

with open("streamlit_app.py", "w") as f:
    f.write(streamlit_app_code)

# Get ngrok authtoken from Colab secrets
NGROK_AUTH_TOKEN = userdata.get('NGROK_AUTH_TOKEN')

# Configure pyngrok
conf.get_default().auth_token = NGROK_AUTH_TOKEN

# Terminate any existing ngrok tunnels
ngrok.kill()

# Start ngrok tunnel for Streamlit on port 8501
public_url = ngrok.connect(8501)
print(f"Streamlit App URL:\n {public_url}")

# Run Streamlit in the background
os.system("nohup streamlit run streamlit_app.py --server.port 8501 > streamlit_output.log 2>&1 &")

print("")
print("Streamlit app is starting. Please open the URL above.")
print("Check 'streamlit_output.log' for logs if it doesn't appear.")


Streamlit App URL:
 NgrokTunnel: "https://algebra-violin-dreamland.ngrok-free.dev" -> "http://localhost:8501"

Streamlit app is starting. Please open the URL above.
Check 'streamlit_output.log' for logs if it doesn't appear.
